# Controlled Task 3 log-target candidate (v13)

Start from the scored v9 combined file and replace **only Task 3** with predictions from `t3_log_positive_amount_full_v6`. Keep Tasks 1, 2, 4, and 5 byte-equivalent as parsed CSV values. This creates a local candidate only; it does not upload to Kaggle or use leaderboard feedback.

In [1]:
from pathlib import Path
import sys,json,hashlib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_model.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.pipeline.random_seed import set_seed
SEED=set_seed()
import numpy as np,pandas as pd
SRC=ROOT/'src';OUT=SRC/'submission';TASK=SRC/'task3'
BASE=OUT/'kaggle_t12345_t5_alert25_v9.csv'
PART=TASK/'submission'/'t3_log_positive_amount_full_v6.csv'
NAME='kaggle_t12345_t3_log_v13'
DEST=OUT/f'{NAME}.csv';META=OUT/f'{NAME}.json'
MODEL=TASK/'save'/'t3_log_positive_amount_full_v6.json'
EVAL=TASK/'save'/'t3_log_positive_amount_full_v6-evaluation.json'
assert BASE.exists() and PART.exists() and MODEL.exists() and EVAL.exists()
assert DEST.exists()==META.exists(),'Inspect incomplete candidate'
sha=lambda p:hashlib.sha256(p.read_bytes()).hexdigest()
base=pd.read_csv(BASE,dtype=str);part=pd.read_csv(PART,sep='|',dtype={'id':str,'task3':float})
base_card=json.loads(BASE.with_suffix('.json').read_text(encoding='utf-8'))
assert base_card['sha256']==sha(BASE)
assert base_card['sources']['task3']['data_name']=='eda_combined_v1'
expected=['id','task1','task2','task3','task4','task5']
assert base.columns.tolist()==expected and part.columns.tolist()==['id','task3']
assert len(base)==len(part)==25889
assert base.id.is_unique and part.id.is_unique
assert base.id.str.match(r'^ANS[0-9]{5}$').all()
data_card=json.loads((TASK/'data'/'eda'/'eda_combined_v1.json').read_text(encoding='utf-8'))
linked_test=pd.read_parquet(SRC/data_card['splits']['test']['path'])
assert part.id.tolist()==linked_test.player_id.tolist()
assert len(linked_test)==len(base)
base_hash=sha(BASE)
print('Base hash:',base_hash,'model:',MODEL.name)

Base hash: 4d4d251d349dd1414a1db524d647b9187101d140f3ee220fffc724b340ad21f4 model: t3_log_positive_amount_full_v6.json


## Replace Task 3 by verified row position and check invariants

Task 3 has player IDs whereas the combined sample has `ANS` IDs. As in the existing assembly notebook, its unchanged linked Test order maps to the sample by row position.

In [2]:
candidate=base.copy()
mapped=part.task3.to_numpy(dtype=float)
assert np.isfinite(mapped).all() and (mapped>=0).all()
candidate['task3']=mapped
assert candidate.columns.tolist()==expected
assert candidate.id.equals(base.id)
for col in ['task1','task2','task4','task5']:
    assert candidate[col].astype(str).equals(base[col].astype(str)),col
changed=int((pd.to_numeric(base.task3)!=mapped).sum())
assert changed>0
if not DEST.exists():
    candidate.to_csv(DEST,index=False)
    detail={'name':NAME,'source_csv':'submission/'+BASE.name,
        'source_sha256':base_hash,'changed_task':'task3','changed_rows':changed,
        'task3_model_card':'task3/save/'+MODEL.name,
        'task3_evaluation_card':'task3/save/'+EVAL.name,
        'task3_draft':'task3/submission/'+PART.name,
        'task3_draft_sha256':sha(PART),'seed':SEED,'rows':len(candidate),
        'task3_id_mapping':'linked EDA Test row position to sample ANS row position, as in assemble_v2',
        'score_status':'unsubmitted','output_sha256':sha(DEST)}
    META.write_text(json.dumps(detail,indent=2)+'\n',encoding='utf-8')
else:
    detail=json.loads(META.read_text(encoding='utf-8'))
    assert detail['source_sha256']==base_hash and detail['task3_draft_sha256']==sha(PART)
    assert detail['output_sha256']==sha(DEST)
    saved=pd.read_csv(DEST,dtype=str)
    assert saved.columns.tolist()==expected and saved.id.equals(base.id)
    for col in ['task1','task2','task4','task5']:
        assert saved[col].equals(base[col]),col
    assert np.allclose(saved.task3.astype(float),mapped)
print('Changed Task 3 rows:',changed,'of',len(candidate))
print('Output hash:',detail['output_sha256'])

Changed Task 3 rows: 25587 of 25889
Output hash: a1a0260892735af3e4a04142f6ca768b12e99fe978c3cf139feba53040006bbe
